# NLP Practical 8 — Machine Translation

Run in **Google Colab**. Cells with `pip install` only need to run once per session.

**Covers:** the MT problem taxonomy, Indian-language morphological characteristics, rule-based MT with reordering, IBM Model 1 word-alignment trained from scratch via EM, the conceptual path to Neural MT, and BLEU evaluation.


In [ ]:
!pip install nltk -q
import nltk
nltk.download("punkt")
from collections import defaultdict


### Problems in Machine Translation (taxonomy)

| Problem | Example |
|---|---|
| **Lexical ambiguity** | "bank" -> financial institution vs. riverbank |
| **Word-order divergence** | English SVO vs. Hindi/Japanese SOV |
| **Morphological richness** | Hindi verb agreement encodes gender+number+honorific; English does not |
| **Idioms / non-compositionality** | "kick the bucket" != literal translation |
| **Free word order + case marking** | Many Indian languages mark grammatical role via case suffixes rather than position |
| **Low-resource pairs** | Few parallel sentences for many Indian language pairs compared to English-French |


In [ ]:
# ============================================================
# PART A: Characteristics of Indian languages relevant to MT
# ============================================================
examples = {
    "Agglutinative morphology": "Hindi 'ladkon-ne' = ladka(boy) + PLURAL + ERGATIVE, one word carries what English needs 3 words for.",
    "SOV word order": "Hindi: 'Ram seb khata hai' (Ram apple eats) vs English SVO: 'Ram eats an apple'.",
    "Postpositions not prepositions": "Hindi uses 'ke upar' (of-top, i.e. 'on top of') after the noun, English uses 'on' before it.",
}
for k, v in examples.items():
    print(f"{k}:\n  {v}\n")


In [ ]:
# ============================================================
# PART B: Rule-Based MT (RBMT) -- dictionary + reordering rule
# ============================================================
dictionary = {"ram": "ram", "eats": "khata hai", "an": "", "apple": "seb"}

def rbmt_translate(sentence):
    words = sentence.lower().replace(".", "").split()
    # naive SVO -> SOV reorder: move the verb to the end
    verb_idx = next((i for i, w in enumerate(words) if w in ("eats", "sees", "buys")), None)
    if verb_idx is not None:
        verb = words.pop(verb_idx)
        words.append(verb)
    return " ".join(dictionary.get(w, w) for w in words if dictionary.get(w, w))

print("EN: Ram eats an apple.")
print("HI (RBMT, SOV-reordered):", rbmt_translate("Ram eats an apple."))


In [ ]:
# ============================================================
# PART C: IBM Model 1 -- word-alignment EM algorithm, trained from scratch
# ============================================================
# Tiny toy parallel corpus (English, French) -- classic IBM Model 1 teaching example
parallel_corpus = [
    (["the", "house"], ["la", "maison"]),
    (["the", "book"], ["le", "livre"]),
    (["a", "book"], ["un", "livre"]),
    (["the", "house", "is", "small"], ["la", "maison", "est", "petite"]),
]

english_vocab = set(w for e, f in parallel_corpus for w in e)
french_vocab = set(w for e, f in parallel_corpus for w in f)

# Initialize t(f|e) uniformly
t = defaultdict(lambda: 1.0 / len(french_vocab))

for iteration in range(20):
    count = defaultdict(float)
    total = defaultdict(float)
    for e_sent, f_sent in parallel_corpus:
        for f in f_sent:
            s_total = sum(t[(f, e)] for e in e_sent)
            for e in e_sent:
                val = t[(f, e)] / s_total
                count[(f, e)] += val
                total[e] += val
    for (f, e), c in count.items():
        t[(f, e)] = c / total[e]

print("Learned translation probabilities t(f|e) after EM (selected entries):")
for e in ["house", "book", "the"]:
    best = sorted(french_vocab, key=lambda f: -t[(f, e)])[:2]
    print(f"  t(f|'{e}') top candidates:", [(f, round(t[(f, e)], 3)) for f in best])

print("\nThis is the exact IBM Model 1 -> Google Translate's pre-2016 SMT approach,")
print("later replaced by Neural MT in 2016 (see next cell).")


### Neural Machine Translation (conceptual) + BLEU evaluation

A full NMT system (encoder-decoder with attention / Transformer) needs GPU training on
millions of sentence pairs, which is out of scope for a single lab cell -- but the
**architecture progression** matters:

```
IBM Model 1 (word alignment, EM)
   -> Phrase-Based SMT (Moses-style)
   -> RNN encoder-decoder with attention (Bahdanau, 2014)
   -> Transformer-based NMT (2017-onward; what powers Google Translate, IndicTrans2, NLLB)
```

Below, BLEU (the standard MT evaluation metric) is computed on the RBMT output from Part B
against a human reference, to show how MT quality is actually scored in practice.


In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

reference = [["ram", "seb", "khata", "hai"]]
candidate = rbmt_translate("Ram eats an apple.").split()

smoothie = SmoothingFunction().method4
score = sentence_bleu(reference, candidate, smoothing_function=smoothie)
print("Candidate translation:", candidate)
print("Reference translation:", reference[0])
print(f"BLEU score: {score:.3f}")

print("\nIndustry note: production MT systems are evaluated with BLEU, chrF, and")
print("increasingly COMET (a learned, embedding-based metric used in WMT shared tasks).")
